# MedSAM — pure C++ on GPU (Colab)

Dependency-free C++ port of **MedSAM** (SAM ViT-B fine-tuned on medical images). No PyTorch at run
time: the ViT-B encoder + SAM decoder compiled with `nvcc -DUSE_CUDA` so matmul/conv GEMMs offload
to **cuBLAS** via the `bk::gemm_hosted` seam. Runtime → Change runtime type → **GPU**.


## 1. GPU + repo


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -2


In [ ]:
%cd /content
![ -d medsam_cpp] || git clone https://github.com/yomei-o/medsam_cpp.git
%cd /content/medsam_cpp


## 2. Weights (one-time extraction — the only Python step)
Downloads the MedSAM ViT-B checkpoint from the official Google Drive folder, then extracts.


In [ ]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git gdown numpy
import gdown, os, glob
os.makedirs('pure/ref', exist_ok=True)
if not glob.glob('pure/ref/**/*.pth', recursive=True):
    gdown.download_folder('https://drive.google.com/drive/folders/1ETWmi4AiniJeWOt6HAsYgTjYv_fkgzoN', output='pure/ref', quiet=True)
src = [p for p in glob.glob('pure/ref/**/*.pth', recursive=True) if 'vit_b' in p or 'flare22' in p][0]
import shutil; shutil.copy(src, 'pure/ref/medsam_vit_b.pth'); print('checkpoint:', src)
!cd pure/ref && python export_medsam.py


## 3. Build — CPU (g++) and GPU (cuBLAS via nvcc -DUSE_CUDA)


In [ ]:
!g++ -O2 -std=c++17 -DNOMINMAX -Ipure/third_party pure/infer_medsam.cpp -o infer_cpu
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Ipure/third_party pure/infer_medsam.cpp -lcublas -o infer_gpu
print('built infer_cpu / infer_gpu')


## 4. Segment a medical image (point prompt) — CPU vs GPU


In [ ]:
import urllib.request, time
urllib.request.urlretrieve('https://raw.githubusercontent.com/bowang-lab/MedSAM/main/assets/img_demo.png','med.png')
import os
for n in ['infer_cpu','infer_gpu']:
    t=time.time(); os.system(f'./{n} med.png --point 256 256 out_{n}.png pure/ref'); print(f'{n:10s} {time.time()-t:5.1f}s')


In [ ]:
from IPython.display import Image, display
display(Image('med.png', width=320), Image('out_infer_gpu.png', width=320))


## 5. Train the mask decoder on GPU (synthetic)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Ipure/third_party pure/train_medsam.cpp -lcublas -o train_gpu
!./train_gpu pure/ref --steps 20
